# Visualizacion Semana 7

Este notebook permite revisar los datos generados por `main.py` desde el CSV exportado y desde MongoDB.

In [ ]:
from pathlib import Path
import os

import certifi
import pandas as pd
from IPython.display import display
from pymongo import MongoClient

PROJECT_DIR = Path('/home/jovyan/work') if Path('/home/jovyan/work').exists() else Path.cwd().resolve().parents[1]
OUTPUT_DIR = PROJECT_DIR / 'semanas' / 'Semana 7 La union' / 'salidas'
CSV_PATH = OUTPUT_DIR / 'union_semana7.csv'
JSON_PATH = OUTPUT_DIR / 'union_semana7.json'

MONGO_URI = os.getenv('MONGODB_URI', 'mongodb://database:27017/')
MONGO_DATABASE = os.getenv('MONGODB_DATABASE', 'proyecto_bigdata')
MONGO_COLLECTION = os.getenv('MONGODB_COLLECTION', 'union_semana7')

print('CSV:', CSV_PATH)
print('JSON:', JSON_PATH)
print('Mongo:', f'{MONGO_DATABASE}.{MONGO_COLLECTION}')

In [ ]:
df = pd.read_csv(CSV_PATH)
print('Filas:', len(df))
print('Columnas:', len(df.columns))
anios_presentes = sorted(df['periodo'].dropna().astype(int).unique().tolist())
print('Anios presentes:', anios_presentes)
assert len(anios_presentes) == 1, 'La comparacion de semana 7 debe usar un solo anio comun.'
display(df.head(10))

In [ ]:
display(df[['integrante', 'dataset', 'indicador', 'pais', 'valor']].sample(10, random_state=7))

In [ ]:
resumen_dataset = (
    df.groupby(['integrante', 'dataset'])['valor']
    .agg(registros='count', promedio='mean', maximo='max')
    .reset_index()
    .sort_values(['integrante', 'registros'], ascending=[True, False])
)

display(resumen_dataset)

In [ ]:
resumen_indicador = (
    df.groupby(['pais', 'indicador', 'categoria_energia'])['valor']
    .agg(registros='count', promedio='mean', maximo='max')
    .reset_index()
    .sort_values('registros', ascending=False)
)

display(resumen_indicador.head(20))

In [ ]:
top_valores = df.sort_values('valor', ascending=False)
display(top_valores[['item', 'pais', 'region', 'dataset', 'indicador', 'valor', 'unidad']].head(20))

In [ ]:
kwargs = {'serverSelectionTimeoutMS': 5000}
if MONGO_URI.startswith('mongodb+srv://'):
    kwargs['tlsCAFile'] = certifi.where()

client = MongoClient(MONGO_URI, **kwargs)
client.admin.command('ping')
coleccion = client[MONGO_DATABASE][MONGO_COLLECTION]

print('Documentos en Mongo:', coleccion.count_documents({}))

In [ ]:
docs = list(coleccion.find({}, {'_id': 0}).limit(10))
display(pd.DataFrame(docs))

In [ ]:
mongo_resumen = list(
    coleccion.aggregate([
        {
            '$group': {
                '_id': {'integrante': '$integrante', 'dataset': '$dataset'},
                'registros': {'$sum': 1},
                'valor_maximo': {'$max': '$valor'}
            }
        },
        {'$sort': {'_id.integrante': 1, 'registros': -1}}
    ])
)

display(pd.json_normalize(mongo_resumen))